In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    when,
    floor,
    concat_ws,
    current_timestamp,
    broadcast
)

catalog = "aircraft_data"
schema = "dataset"

# Bronze Tables
bronze_table = f"{catalog}.{schema}.bronze_flight_data"
airline_reference_table = f"{catalog}.{schema}.aircraft_table"

# Audit Tables
bronze_audit_table = f"{catalog}.{schema}.pipeline_audit"


# Silver Tables
silver_table = f"{catalog}.{schema}.silver_flight_data"


bronze_df = spark.table(bronze_table)
airline_df = spark.table(airline_reference_table)
silver_df = bronze_df
silver_df = spark.table(silver_table)


In [0]:
'''from pyspark.sql.types import *

audit_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("file_year", IntegerType(), True),
    StructField("file_month", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("records_loaded", IntegerType(), True),
    StructField("processed_timestamp", TimestampType(), True)
])

empty_df = spark.createDataFrame([], audit_schema)

(
    empty_df.write
        .format("delta")
        .mode("ignore")
        .saveAsTable(silver_audit_table)
)'''

In [0]:
'''# ==========================================================
# STEP 2 : Read Incremental Bronze Data
# ==========================================================

from pyspark.sql.utils import AnalysisException

# Read Bronze Flight Data
bronze_df = spark.table(bronze_table)

# Read Airline Reference Table
airline_df = spark.table(airline_reference_table)

# ----------------------------------------------------------
# First Run Check
# ----------------------------------------------------------

try:

    silver_partitions = (
        spark.table(silver_table)
        .select("Year", "Month")
        .distinct()
    )

    # Find Bronze partitions that do not exist in Silver
    new_partitions = (
        bronze_df
        .select("Year", "Month")
        .distinct()
        .join(
            silver_partitions,
            ["Year", "Month"],
            "left_anti"
        )
    )

    # Read only new Bronze partitions
    bronze_df = (
        bronze_df.alias("b")
        .join(
            new_partitions.alias("p"),
            ["Year", "Month"],
            "inner"
        )
    )

    print("Incremental Load")

except AnalysisException:

    print("First Silver Load")

# Check records to process
print("Rows To Process :", bronze_df.count())

display(
    bronze_df
    .select("Year", "Month")
    .distinct()
    .orderBy("Year", "Month")
)'''

In [0]:
# ==========================================================
# STEP 2 : Read Incremental Bronze Data
# ==========================================================

from pyspark.sql.utils import AnalysisException

bronze_df = spark.table(bronze_table)
airline_df = spark.table(airline_reference_table)

try:

    silver_partitions = (
        spark.table(silver_table)
        .select("Year", "Month")
        .distinct()
    )

    new_partitions = (
        bronze_df
        .select("Year", "Month")
        .distinct()
        .join(
            silver_partitions,
            ["Year", "Month"],
            "left_anti"
        )
    )

    bronze_df = (
        bronze_df.alias("b")
        .join(
            new_partitions.alias("p"),
            ["Year", "Month"],
            "inner"
        )
    )

except AnalysisException:
    pass

In [0]:
from pyspark.sql.utils import AnalysisException

try:
    rows_to_process = bronze_df.count()
except AnalysisException:
    bronze_df = spark.table(bronze_table)
    rows_to_process = bronze_df.count()

print(f"Rows to Process : {rows_to_process}")


In [0]:
if rows_to_process >0:
    silver_df = bronze_df
    silver_df = silver_df.dropDuplicates([
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime"])
    
    critical_columns = [
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest"]


    silver_df = silver_df.dropna(
        subset=critical_columns
    )

    cols_to_drop = [
    "_c109",
    "Div4Airport","Div4AirportID","Div4AirportSeqID","Div4WheelsOn",
    "Div4TotalGTime","Div4LongestGTime","Div4WheelsOff","Div4TailNum",
    "Div5Airport","Div5AirportID","Div5AirportSeqID","Div5WheelsOn",
    "Div5TotalGTime","Div5LongestGTime","Div5WheelsOff","Div5TailNum",   "FirstDepTime",
    "TotalAddGTime",
    "LongestAddGTime",
    "DivReachedDest",
    "DivActualElapsedTime",
    "DivArrDelay",
    "DivDistance",
    "Div1Airport",
    "Div1AirportID",
    "Div1AirportSeqID",
    "Div1WheelsOn",
    "Div1TotalGTime",
    "Div1LongestGTime",
    "Div1WheelsOff",
    "Div1TailNum",
    "Div2Airport",
    "Div2AirportID",
    "Div2AirportSeqID",
    "Div2WheelsOn",
    "Div2TotalGTime",
    "Div2LongestGTime",
    "Div2WheelsOff",
    "Div2TailNum",
    "Div3Airport",
    "Div3AirportID",
    "Div3AirportSeqID",
    "Div3WheelsOn",
    "Div3TotalGTime",
    "Div3LongestGTime",
    "Div3WheelsOff",
    "Div3TailNum"]

    silver_df = silver_df.drop(*cols_to_drop)


    #checking the invalid airport list
    silver_df = silver_df.filter(
    (F.col("Origin").rlike("^[A-Z]{3}$")) &
    (F.col("Dest").rlike("^[A-Z]{3}$")) )
    
    #negative distance
    silver_df = silver_df.filter(
    F.col("Distance") >= 0)

    #look for the negative airtime
    silver_df = silver_df.filter(
    (F.col("AirTime").isNull()) |
    (F.col("AirTime") >= 0))

    #compare the time of the flight with current time
    silver_df = silver_df.filter(
    F.col("FlightDate") <= F.current_date())

    #categorize the flight delays
    silver_df = silver_df.withColumn(
    "DelayCategory",when(silver_df.ArrDelay.isNull(), "Not Available")
    .when(silver_df.ArrDelay <= 0, "On Time")
    .when(silver_df.ArrDelay <= 15, "0-15 Minutes")
    .when(silver_df.ArrDelay <= 30, "16-30 Minutes")
    .when(silver_df.ArrDelay <= 60, "31-60 Minutes")
    .otherwise("60+ Minutes"))

    #Dealy Reason
    silver_df = silver_df.withColumn(
    "PrimaryDelayReason",
    when(silver_df.CarrierDelay > 0, "Carrier")
    .when(silver_df.WeatherDelay > 0, "Weather")
    .when(silver_df.NASDelay > 0, "NAS")
    .when(silver_df.SecurityDelay > 0, "Security")
    .when(silver_df.LateAircraftDelay > 0, "Late Aircraft")
    .otherwise("No Delay"))

    #flight performance
    silver_df = silver_df.withColumn(
    "FlightPerformance",
    when(silver_df.Cancelled == 1, "Cancelled")
    .when(silver_df.Diverted == 1, "Diverted")
    .when(silver_df.ArrDelay <= 0, "Early")
    .when(silver_df.ArrDelay <= 15, "On Time")
    .otherwise("Delayed"))
    
    #Arrival Delay Bucket
    silver_df = silver_df.withColumn(
    "DelayBucket",
    when(silver_df.ArrDelayMinutes <= 0, "On Time")
    .when(silver_df.ArrDelayMinutes <= 15, "0-15 Minutes")
    .when(silver_df.ArrDelayMinutes <= 30, "16-30 Minutes")
    .when(silver_df.ArrDelayMinutes <= 60, "31-60 Minutes")
    .otherwise("60+ Minutes"))
    
    #flight distance
    silver_df = silver_df.withColumn(
    "FlightDistance",
    when(silver_df.Distance <= 500, "Short")
    .when(silver_df.Distance <= 1000, "Medium")
    .otherwise("Long"))
    
    #compare weekdays and weekends
    silver_df = silver_df.withColumn(
    "DayType",
    when(silver_df.DayOfWeek.isin(6,7), "Weekend")
    .otherwise("Weekday"))
    
    #creating the seasonal data
    silver_df = silver_df.withColumn(
    "Season",
    when(silver_df.Month.isin(12,1,2), "Winter")
    .when(silver_df.Month.isin(3,4,5), "Spring")
    .when(silver_df.Month.isin(6,7,8), "Summer")
    .otherwise("Fall"))

    #dep time bucket
    silver_df = silver_df.withColumn(
    "DepartureHour",
    floor(silver_df.CRSDepTime / 100))

    #merging the flight routes

    silver_df = silver_df.withColumn(
    "Route",
    concat_ws(" -> ", silver_df.Origin, silver_df.Dest))
    
    #connecting the airline name with the code accordingly
    silver_df = (
    silver_df.alias("f")
    .join(
        airline_df.alias("a"),
        F.col("f.Reporting_Airline") == F.col("a.Code"),
        "left"
    )
    .select(
        "f.*",
        F.col("a.Description").alias("Airline_Name")
    )) 

    silver_df = (
    silver_df
    .withColumn(
        "silver_processed_timestamp",
        current_timestamp()
    ))

    #writing the updated data into table with the partition
    (
    silver_df.write
      .format("delta")
      .mode("overwrite")
      .option("mergeSchema","true")
      .partitionBy("Year", "Month")
      .saveAsTable(silver_table))
   

    
else:


    print("No new data. Silver transformation skipped.")

print("Silver table updated successfully.")

In [0]:
display(
    spark.table(silver_table).limit(10)
)